# novel-ocr : 소설 스캔 이미지 → 텍스트 추출

Colab에서 실행하는 용도의 노트북입니다. 순서대로 셀을 실행하세요 (Shift+Enter).

**사용 순서**
1. 런타임을 GPU로 설정 (상단 메뉴: 런타임 > 런타임 유형 변경 > T4 GPU)
2. 아래 셀을 위에서부터 순서대로 실행
3. Google Drive에 스캔 이미지를 올려두고 경로만 맞춰주면 됨
4. 결과 텍스트 파일이 Drive에 자동 저장됨

## 1. 라이브러리 설치

In [ ]:
!pip install easyocr -q

## 2. GPU 확인
출력이 `False`로 나오면 런타임 유형이 GPU로 설정되지 않은 것입니다. 상단 메뉴에서 런타임 > 런타임 유형 변경 > T4 GPU로 바꾸고 다시 실행하세요.

In [ ]:
import torch
print("GPU 사용 가능:", torch.cuda.is_available())

## 3. Google Drive 연동
실행하면 인증 창이 뜹니다. 로그인 후 권한을 허용하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 경로 설정

미리 Google Drive에 스캔 이미지를 업로드해두고, 그 폴더 경로를 아래에 적어주세요.

예: Drive의 `내 드라이브 > novel_scans` 폴더에 이미지를 올렸다면
`/content/drive/MyDrive/novel_scans`

In [ ]:
IMAGE_FOLDER = '/content/drive/MyDrive/novel_scans'   # 스캔 이미지가 있는 폴더
OUTPUT_FOLDER = '/content/drive/MyDrive/novel_ocr_output'  # 결과 저장 폴더
OUTPUT_FILE = f'{OUTPUT_FOLDER}/novel_text.txt'

import os
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print('이미지 폴더:', IMAGE_FOLDER)
print('결과 파일:', OUTPUT_FILE)

## 5. OCR 리더 준비
처음 실행 시 모델을 다운로드하기 때문에 1~2분 걸릴 수 있습니다.

In [ ]:
import easyocr

reader = easyocr.Reader(['ko'], gpu=True)  # 영어가 섞여 있으면 ['ko', 'en']

## 6. 전체 이미지 일괄 처리

파일명 순서대로 처리하고, 파일 하나 끝날 때마다 바로 Drive에 저장합니다.
(중간에 세션이 끊겨도 그때까지 처리한 내용은 Drive에 남아있음)

In [ ]:
import glob

# 필요하면 확장자 추가/수정 (jpg, jpeg, png 등)
image_files = sorted(
    glob.glob(os.path.join(IMAGE_FOLDER, '*.jpg')) +
    glob.glob(os.path.join(IMAGE_FOLDER, '*.jpeg')) +
    glob.glob(os.path.join(IMAGE_FOLDER, '*.png'))
)

print(f'총 {len(image_files)}개 이미지 발견')

# 이미 처리된 파일은 건너뛰기 위한 체크 (재실행 시 중복 방지)
done_markers = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        content = f.read()
    for line in content.splitlines():
        if line.startswith('--- ') and line.endswith(' ---'):
            done_markers.add(line[4:-4])

with open(OUTPUT_FILE, 'a', encoding='utf-8') as f:
    for i, img_path in enumerate(image_files):
        fname = os.path.basename(img_path)
        if fname in done_markers:
            continue  # 이미 처리됨
        result = reader.readtext(img_path, detail=0, paragraph=True)
        text = '\n'.join(result)
        f.write(f"\n\n--- {fname} ---\n\n")
        f.write(text)
        f.flush()
        print(f'[{i+1}/{len(image_files)}] 처리 완료: {fname}')

print('\n모든 작업 완료! 결과 파일:', OUTPUT_FILE)

## 7. 결과 미리보기

In [ ]:
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    preview = f.read()
print(preview[:2000])